# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalahmed251/-ML-Search-Discovery/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# I will use a Random Forest classifier because the target is binary: declining or not declining.
# Random Forest can learn non-linear relationships between search-performance features and page decline.
# It also provides feature importance, which helps us interpret which signals the model uses.
# The model will be evaluated as a ranking system using Precision@50, using the same comparison logic as the Week-4 hand-written baseline.

In [5]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

repo = "/content/ML-Search-Discovery"

if not os.path.exists(repo):
    !git clone -q --depth 1 https://github.com/bilalahmed251/-ML-Search-Discovery.git {repo}

df = pd.read_csv(
    f"{repo}/data/raw/content_refresh_anonymized.csv"
 ).copy()

# Binary target: 1 = declining, 0 = not declining
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
]

categorical_features = [
    "content_type",
    "main_intent",
    "position_tier",
    "impression_tier",
]

numeric_features = [
    col for col in numeric_features if col in df.columns
]

categorical_features = [
    col for col in categorical_features if col in df.columns
]

X_numeric = df[numeric_features].apply(
    pd.to_numeric,
    errors="coerce"
).copy()

X_numeric = X_numeric.fillna(X_numeric.median())

X_categorical = pd.get_dummies(
    df[categorical_features].fillna("MISSING"),
    columns=categorical_features,
    dtype=int
)

X = pd.concat(
    [X_numeric, X_categorical],
    axis=1
)

y = df["is_declining_label"]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Rows:", len(df))
print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
print("Model:", model)

Rows: 30000
Feature matrix shape: (30000, 27)
Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Model: RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# The dataset is a single 90-day page-level snapshot and does not provide a reliable client or observation-date field for a grouped or time-aware split.
# Therefore, I will use a reproducible stratified holdout split: 80% for training and 20% for testing. Stratification keeps the declining/non-declining label ratio similar in both sets. The test set will remain untouched until final evaluation.


In [7]:
from sklearn.model_selection import train_test_split

# Keep the same row indices for X and y
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training declining rate:", round(y_train.mean(), 3))
print("Testing declining rate:", round(y_test.mean(), 3))


Training rows: 24000
Testing rows: 6000
Training declining rate: 0.542
Testing declining rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# I will train the Random Forest on the training set only and evaluate it on the untouched test set.
# Both the Random Forest and the hand-written baseline will rank the same test pages.
# I will compare them using Precision@50: among the 50 pages ranked highest for refresh, how many are actually declining according to the observed label.


In [9]:
import numpy as np
from sklearn.metrics import precision_score

# Train only on the training data
model.fit(X_train, y_train)

# Random Forest probability for class 1 = declining
model_scores = model.predict_proba(X_test)[:, 1]

# Helper function for ranking evaluation
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    top_k_indices = np.argsort(scores)[::-1][:k]
    return float(y_true[top_k_indices].mean())

# Random Forest Precision@50
model_p50 = precision_at_k(y_test.to_numpy(), model_scores, k=50)

# Rebuild the Week-4 hand-written baseline on exactly the same test rows
baseline_test = df.loc[X_test.index].copy()

baseline_numeric = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]

for col in baseline_numeric:
    baseline_test[col] = pd.to_numeric(
        baseline_test[col],
        errors="coerce"
    )

baseline_test = baseline_test.dropna(
    subset=baseline_numeric + ["is_declining_label"]
).copy()

# Percentile-based hand-written baseline components
baseline_test["visibility_score"] = (
    baseline_test["impressions_90d"].rank(pct=True)
)
baseline_test["age_score"] = (
    baseline_test["content_age_days"].rank(pct=True)
)
baseline_test["stale_score"] = (
    baseline_test["days_since_last_update"].rank(pct=True)
)
baseline_test["position_score"] = (
    baseline_test["avg_position"].rank(pct=True)
)
baseline_test["low_ctr_score"] = (
    1 - baseline_test["ctr"].rank(pct=True)
)

baseline_test["baseline_score"] = (
    0.30 * baseline_test["visibility_score"] +
    0.20 * baseline_test["age_score"] +
    0.20 * baseline_test["stale_score"] +
    0.15 * baseline_test["position_score"] +
    0.15 * baseline_test["low_ctr_score"]
)

baseline_p50 = precision_at_k(
    baseline_test["is_declining_label"].to_numpy(),
    baseline_test["baseline_score"].to_numpy(),
    k=50
)

comparison = pd.DataFrame({
    "method": [
        "Week-4 hand-written baseline",
        "Week-5 Random Forest"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

display(comparison)

print("Random Forest Precision@50:", round(model_p50, 3))
print("Baseline Precision@50:", round(baseline_p50, 3))


,method,precision_at_50
0,Week-4 hand-written baseline,0.70
1,Week-5 Random Forest,0.94


Random Forest Precision@50: 0.94
Baseline Precision@50: 0.7


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# I will inspect false positives, false negatives, and feature importance.
# False positives are pages the model ranked highly even though they were not labeled declining.
# False negatives are declining pages that the model did not rank highly.
# Feature importance shows which available signals the Random Forest used most strongly, but it does not prove causation.


In [11]:
# Predictions on the untouched test set
test_results = df.loc[X_test.index, [
    "content_id",
    "trend_direction",
    "is_declining_label"
]].copy()

test_results["model_score"] = model_scores
test_results["predicted_label"] = (model_scores >= 0.5).astype(int)

# Error types
test_results["error_type"] = "correct"
test_results.loc[
    (test_results["is_declining_label"] == 0) &
    (test_results["predicted_label"] == 1),
    "error_type"
] = "false_positive"

test_results.loc[
    (test_results["is_declining_label"] == 1) &
    (test_results["predicted_label"] == 0),
    "error_type"
] = "false_negative"

print("Error counts:")
print(test_results["error_type"].value_counts())

print("\nExample false positives:")
display(
    test_results[
        test_results["error_type"] == "false_positive"
    ].sort_values("model_score", ascending=False).head(10)
)

print("\nExample false negatives:")
display(
    test_results[
        test_results["error_type"] == "false_negative"
    ].sort_values("model_score", ascending=True).head(10)
)

# Feature importance
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop features used by the model:")
display(feature_importance.head(10))


Error counts:
error_type
correct           4148
false_positive    1097
false_negative     755
Name: count, dtype: int64

Example false positives:


,content_id,trend_direction,is_declining_label,model_score,predicted_label,error_type
5844,content_425b44a2315e,stable,0,0.960,1,false_positive
14549,content_d020d42e7fcc,up,0,0.960,1,false_positive
2164,content_1337de8128cc,up,0,0.950,1,false_positive
13651,content_e193494bafc7,stable,0,0.945,1,false_positive
12983,content_e913ab1f851a,stable,0,0.940,1,false_positive
7882,content_25a763874cf0,up,0,0.935,1,false_positive
21209,content_da9b0ec85c3d,stable,0,0.935,1,false_positive
8990,content_a38c8f61e246,stable,0,0.930,1,false_positive
24428,content_b178d2c17955,stable,0,0.925,1,false_positive
28018,content_f548487c7b11,stable,0,0.920,1,false_positive



Example false negatives:


,content_id,trend_direction,is_declining_label,model_score,predicted_label,error_type
3700,content_c62f296c7eff,down,1,0.035,0,false_negative
6727,content_eae6a30bae44,down,1,0.035,0,false_negative
11300,content_bd91e21791bb,down,1,0.035,0,false_negative
23748,content_c0af3d6f9dd3,down,1,0.035,0,false_negative
29579,content_5a8e6c869488,down,1,0.060,0,false_negative
5608,content_a55d958ec725,down,1,0.060,0,false_negative
11058,content_1a0ba11dff95,down,1,0.075,0,false_negative
26585,content_0a47afb8d5ae,down,1,0.075,0,false_negative
16182,content_51d78474a788,down,1,0.090,0,false_negative
24852,content_59cab09d0b52,down,1,0.090,0,false_negative



Top features used by the model:


,feature,importance
0,impressions_90d,0.165541
4,avg_position,0.144203
6,content_age_days,0.128576
5,word_count,0.104524
2,sessions_90d,0.086742
9,scroll_rate,0.071388
3,ctr,0.066493
1,clicks_90d,0.051921
7,days_since_last_update,0.039306
8,engagement_rate,0.034802


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.